Sel 1 — Instal Faker

In [1]:
!pip install faker -q

Sel 2 — K-1. Import library

In [2]:
import numpy as np
import pandas as pd
from faker import Faker
import random

print("pandas version:", pd.__version__)

pandas version: 2.2.3


Sel 3 — K-2. Membuat dataset sintetis

In [3]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + "  "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])  # rating opsional

    rows.append({
        "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
        "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
        "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))
df.head()

Jumlah baris: 515


,transaction_id,customer_name,product_name,category,price,quantity,payment_method,transaction_date,shipping_city,rating
0,TRX00305,Ophelia Hartati,Quo Basic,Buku,50000,4,kartu kredit,2026-08-23,Blitar,1.0
1,TRX00500,Cut Maya Wijayanti,Delectus,Elektronik,Rp500.000,1,E-Wallet,2026-08-19,Lubuklinggau,NaN
2,TRX00442,"Dr. Ridwan Utama, M.Pd",Animi Max,Elektronik,25000,1,COD,2026-10-05,Probolinggo,2.0
3,TRX00154,"Viman Suwarno, S.H.",Consectetur,Fashion,Rp250.000,1,COD,06/09/2026,Tual,4.0
4,TRX00074,NaN,Eius,Olahraga,250000.0,4,NaN,11/08/2026,NaN,1.0


Sel 4 — K-3. Deteksi missing value

In [4]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64


Sel 5 — K-3. Penanganan missing value

In [5]:
df = df.dropna(subset=["customer_name", "payment_method"]).copy()
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")

print("Jumlah baris setelah dropna:", len(df))
print(df.isnull().sum())

Jumlah baris setelah dropna: 495
transaction_id        0
customer_name         0
product_name          0
category              0
price                 0
quantity              0
payment_method        0
transaction_date      0
shipping_city         0
rating              158
dtype: int64


Sel 6 — K-4. Deteksi & penanganan duplicate


In [6]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


Sel 7 — K-5a. Standardisasi teks kategorikal



In [7]:
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan; kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

print(df["category"].unique())
print(df["payment_method"].unique())

<StringArray>
['Buku', 'Elektronik', 'Fashion', 'Rumah Tangga', 'Kesehatan', 'Olahraga']
Length: 6, dtype: string
<StringArray>
['Kartu Kredit', 'E-Wallet', 'COD', 'Transfer Bank']
Length: 4, dtype: string


Sel 8 — K-5b. Koreksi tipe data price

In [8]:
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)
df["price"].describe()

,price
count,4.900000e+02
mean,7.504898e+05
std,1.902742e+06
min,1.500000e+04
25%,5.000000e+04
50%,1.500000e+05
75%,5.000000e+05
max,1.200000e+07


Sel 9 — K-5c. Standardisasi tanggal

In [9]:
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")
df["transaction_date"].head()


,transaction_date
0,2026-08-23
1,2026-08-19
2,2026-10-05
3,2026-09-06
5,2026-10-26


Sel 10 — K-5d. Finalisasi tipe data

In [10]:
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

print(df.dtypes)
print("Nilai unik category:", df["category"].nunique())
print("Nilai unik payment_method:", df["payment_method"].nunique())

transaction_id              object
customer_name               object
product_name                object
category            string[python]
price                      float64
quantity                     int64
payment_method      string[python]
transaction_date            object
shipping_city       string[python]
rating                     float64
dtype: object
Nilai unik category: 6
Nilai unik payment_method: 4


Sel 11 — K-6. Ekspor dataset bersih

In [11]:
df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris


Sel 12 — Latihan 1 (SEED = 7)
Pipeline dibungkus fungsi supaya df hasil SEED 42 tidak tertimpa.

In [12]:
def jalankan_pipeline(seed):
    np.random.seed(seed); random.seed(seed)
    fake = Faker("id_ID"); Faker.seed(seed)
    rows = []
    for i in range(1, N + 1):
        trx_id = f"TRX{i:05d}"
        nama_pelanggan = fake.name()
        produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
        kategori = random.choice(kategori_produk)
        harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
        qty = random.randint(1, 5)
        harga = random.choice([str(harga_dasar), f"Rp{harga_dasar:,}".replace(",", "."),
                               f"{harga_dasar}.0", f" {harga_dasar} "])
        tgl = fake.date_between(start_date="-90d", end_date="today")
        tanggal = random.choice([tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")])
        metode = random.choice(metode_bayar)
        if random.random() < 0.3: metode = metode.lower()
        if random.random() < 0.2: kategori = kategori.upper() + "  "
        kota = fake.city()
        rating = random.choice([1, 2, 3, 4, 5, None, None])
        rows.append({"transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
                     "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
                     "transaction_date": tanggal, "shipping_city": kota, "rating": rating})
    d = pd.DataFrame(rows)
    for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
        d.loc[d.sample(frac=frac, random_state=seed).index, col] = np.nan
    d = pd.concat([d, d.sample(n=15, random_state=seed)], ignore_index=True)
    d = d.sample(frac=1, random_state=seed).reset_index(drop=True)
    n_mentah = len(d)

    d = d.dropna(subset=["customer_name", "payment_method"]).copy()
    d["shipping_city"] = d["shipping_city"].fillna("Tidak Diketahui")
    d = d.drop_duplicates()
    for col in ["category", "payment_method", "shipping_city"]:
        d[col] = d[col].astype("string").str.strip().str.title()
    d["payment_method"] = d["payment_method"].replace({"Cod": "COD"})
    d["price"] = d["price"].apply(bersihkan_harga).astype(float)
    d["transaction_date"] = d["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")
    d["quantity"] = d["quantity"].astype(int)
    return n_mentah, len(d)

for s in [42, 7]:
    mentah, bersih = jalankan_pipeline(s)
    print(f"SEED={s:>2} | mentah: {mentah} baris | bersih: {bersih} baris | dibuang: {mentah - bersih}")

SEED=42 | mentah: 515 baris | bersih: 490 baris | dibuang: 25
SEED= 7 | mentah: 515 baris | bersih: 490 baris | dibuang: 25


Sel 13 — Latihan 2 (is_valid_price)

In [13]:
df["is_valid_price"] = df["price"] > 0
print(df["is_valid_price"].value_counts())
print("Jumlah harga tidak valid:", (~df["is_valid_price"]).sum())



is_valid_price
True    490
Name: count, dtype: int64
Jumlah harga tidak valid: 0


Sel 14 — Latihan 3 (transaksi per category)

In [14]:
print(df["category"].value_counts())

category
Olahraga        97
Kesehatan       91
Elektronik      89
Buku            82
Fashion         66
Rumah Tangga    65
Name: count, dtype: Int64


A. KODE LATIHAN

In [15]:

# LATIHAN 1 — Ubah SEED menjadi 7, bandingkan dengan SEED 42

def jalankan_pipeline(seed):
    np.random.seed(seed); random.seed(seed)
    fake = Faker("id_ID"); Faker.seed(seed)
    rows = []
    for i in range(1, N + 1):
        trx_id = f"TRX{i:05d}"
        nama_pelanggan = fake.name()
        produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
        kategori = random.choice(kategori_produk)
        harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
        qty = random.randint(1, 5)
        harga = random.choice([str(harga_dasar), f"Rp{harga_dasar:,}".replace(",", "."),
                               f"{harga_dasar}.0", f" {harga_dasar} "])
        tgl = fake.date_between(start_date="-90d", end_date="today")
        tanggal = random.choice([tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")])
        metode = random.choice(metode_bayar)
        if random.random() < 0.3: metode = metode.lower()
        if random.random() < 0.2: kategori = kategori.upper() + "  "
        kota = fake.city()
        rating = random.choice([1, 2, 3, 4, 5, None, None])
        rows.append({"transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
                     "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
                     "transaction_date": tanggal, "shipping_city": kota, "rating": rating})
    d = pd.DataFrame(rows)
    for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
        d.loc[d.sample(frac=frac, random_state=seed).index, col] = np.nan
    d = pd.concat([d, d.sample(n=15, random_state=seed)], ignore_index=True)
    d = d.sample(frac=1, random_state=seed).reset_index(drop=True)
    n_mentah = len(d)
    n_rating_kosong = d["rating"].isna().sum()

    d = d.dropna(subset=["customer_name", "payment_method"]).copy()
    d["shipping_city"] = d["shipping_city"].fillna("Tidak Diketahui")
    d = d.drop_duplicates()
    return n_mentah, len(d), n_rating_kosong

for s in [42, 7]:
    mentah, bersih, rating_kosong = jalankan_pipeline(s)
    print(f"SEED={s:>2} | mentah: {mentah} | bersih: {bersih} | rating kosong (mentah): {rating_kosong}")

SEED=42 | mentah: 515 | bersih: 490 | rating kosong (mentah): 166
SEED= 7 | mentah: 515 | bersih: 490 | rating kosong (mentah): 120


In [16]:

# LATIHAN 2 — Kolom is_valid_price (True jika price > 0)

df["is_valid_price"] = df["price"] > 0
print(df["is_valid_price"].value_counts())
print("Jumlah harga tidak valid:", (~df["is_valid_price"]).sum())

is_valid_price
True    490
Name: count, dtype: int64
Jumlah harga tidak valid: 0


In [17]:

# LATIHAN 3 — Jumlah transaksi per category (dataset bersih)

print(df["category"].value_counts())
print("Total:", df["category"].value_counts().sum())

category
Olahraga        97
Kesehatan       91
Elektronik      89
Buku            82
Fashion         66
Rumah Tangga    65
Name: count, dtype: Int64
Total: 490


In [18]:

# TAMBAHAN — Bukti bug harga ".0" (untuk bagian Temuan)

mentah_str = pd.read_csv("transaksi_mentah.csv", dtype={"price": str})
contoh = mentah_str[mentah_str["price"].str.endswith(".0")]["price"].head(5)
for x in contoh:
    print(repr(x), "->", bersihkan_harga(x))
print("Harga maksimum di data bersih:", df["price"].max())

'250000.0' -> 2500000.0
'15000.0' -> 150000.0
'15000.0' -> 150000.0
'120000.0' -> 1200000.0
'75000.0' -> 750000.0
Harga maksimum di data bersih: 12000000.0


In [19]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
